# Глава 12. Временные ряды — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Временной ряд: индекс-дата, resample и скользящие окна

Временной ряд — значения, упорядоченные во времени: заказы по часам, выручка по дням. У него свои инструменты: дата как индекс таблицы, переход между частотами (часы → дни → недели), скользящие средние и сдвиги. Осваиваем их на заказах через приложение «Зерна».

Теория — на сайте курса: `ch12/01-resample.html`. Задачи «собери код из строк» (1) остались на сайте.

### Ряд из файла · разминка

Загрузите `app_orders.csv` как Series `s` с индексом-датой. Сохраните `n` — число часов в ряду, `start` и `end` — первую и последнюю дату (значения индекса), `zero_hours` — сколько часов с нулём заказов.

*Подсказки (2): `dsc.hint('ch12/01-resample/load')`*

In [ ]:
%%task ch12/01-resample/load
import pandas as pd

### Один день акции · разминка

Ряд `s` загружен в заготовке. Сохраните `day_total` — сколько заказов было 22 мая 2026 года за весь день, `morning` — с 08:00 до 10:00 включительно и `share_morning` — долю утра в дне (3 знака).

*Подсказки (2): `dsc.hint('ch12/01-resample/slice')`*

In [ ]:
%%task ch12/01-resample/slice
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Дни и недели · разминка

Сохраните `daily` — заказы по дням (сумма) и `weekly_mean` — среднее число заказов в день по неделям (сначала дни, потом среднее по неделям: `daily.resample("W").mean()`). В `best_week` — дату конца недели с наибольшим средним.

*Подсказки (2): `dsc.hint('ch12/01-resample/daily')`*

In [ ]:
%%task ch12/01-resample/daily
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Суточный профиль · база

Посчитайте средний заказ по часу суток отдельно для будней и выходных: DataFrame `profile` (индекс — час 0–23, столбцы `будни` и `выходные`, 1 знак). Сохраните `peak_weekday`, `peak_weekend` — час максимума в каждом столбце.

*Подсказки (2): `dsc.hint('by-hour')`*

In [ ]:
%%task ch12/01-resample/by-hour
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Рост по скользящему среднему · база

По дневному ряду посчитайте 28-дневное скользящее среднее `trend28`. Сохраните `first_value` — его первое непустое значение, `last_value` — последнее (оба целые, округлением) и `growth_pct` — рост в процентах (1 знак).

*Подсказки (2): `dsc.hint('ch12/01-resample/growth')`*

In [ ]:
%%task ch12/01-resample/growth
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
daily = s.resample("D").sum()

### Рост по месяцам · база

Посчитайте `monthly` — сумму заказов по месяцам (`"MS"`) и `growth` — рост к предыдущему месяцу в процентах (`pct_change() * 100`, 1 знак). Месяцы разной длины, поэтому посчитайте и `per_day` — среднее число заказов в день по месяцам (1 знак) и `growth_per_day` — его рост в процентах (1 знак). В `slowest_month` — месяц (число 1–12) с наименьшим ростом `growth`, а в `slowest_month_per_day` — с наименьшим ростом `growth_per_day`.

*Подсказки (3): `dsc.hint('ch12/01-resample/months')`*

In [ ]:
%%task ch12/01-resample/months
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Ночь или сбой · посложнее

Нули в часовом ряду бывают двух видов: ночью заказов просто нет, а днём ноль — подозрение на сбой. Посчитайте `zero_by_hour` — долю нулевых часов для каждого часа суток (Series с индексом 0–23, 2 знака) и `night_hours` — список часов, где эта доля больше 0.3. Затем найдите **дневные** нули — часы с 8 до 20 включительно с нулём заказов: `n_day_zeros` — их число и `outage_dates` — отсортированный список дат (`datetime.date`), когда они были.

*Подсказки (3): `dsc.hint('zeros')`*

In [ ]:
%%task ch12/01-resample/zeros
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Поиск аномальных часов · посложнее

Найдите часы, где заказов намного меньше обычного для этого часа и типа дня. «Обычное» — медиана заказов за этот час суток в этот же день недели за 4 предыдущие недели: `s.shift(168 * k)` для k = 1…4, медиана по строке. Отметьте часы с 10 до 20 включительно, где обычное значение не меньше 8, а фактическое меньше 20 % обычного. Сохраните список `bad_hours` (значения индекса) и `bad_days` — отсортированный список уникальных дат этих часов (как `datetime.date`).

*Подсказки (2): `dsc.hint('anomalies')`*

In [ ]:
%%task ch12/01-resample/anomalies
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Почему пятницы · со звёздочкой

Пятницы выделяются: посчитайте среднее число заказов в день по дням недели (`daily.groupby(daily.index.dayofweek).mean()`) и сохраните `dow_mean` (Series, 1 знак). Затем найдите 5 дней с наибольшим отношением «заказы / 7-дневное скользящее среднее, сдвинутое на 1 день» (`daily.shift(1).rolling(7).mean()`) — это всплески относительно недавнего уровня. Сохраните их даты `spikes` (список `datetime.date` в порядке убывания отношения) и `all_fridays` — `True`, если все они пятницы.

*Подсказки (2): `dsc.hint('friday')` · задача по желанию*

In [ ]:
%%task ch12/01-resample/friday
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
daily = s.resample("D").sum()

## Урок 2. Тренд, сезонность и остаток

Любой ряд удобно представить как сумму трёх частей: медленный тренд, повторяющаяся сезонность и всё остальное — остаток. Раскладываем заказы на части, смотрим на автокорреляцию — «похожесть ряда на самого себя со сдвигом» — и разбираемся, что такое стационарность и зачем её добиваются разностями.

Теория — на сайте курса: `ch12/02-decompose.html`. Задачи «собери код из строк» (1) остались на сайте.

### Разложение дневного ряда · разминка

Разложите дневные заказы с `period=7` и сохраните `trend_growth` — разницу между последним и первым непустыми значениями тренда (1 знак) и `best_dow` — номер дня недели (0 — понедельник) с наибольшей сезонной поправкой.

*Подсказки (2): `dsc.hint('decompose')`*

In [ ]:
%%task ch12/02-decompose/decompose
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
daily = s.resample("D").sum()

### Тренд скользящим средним · разминка

В заготовке — ряд `toy` из 21 дня: тренд «50 + 3 в день» и недельный узор. Посчитайте `trend7` — центрированное скользящее среднее за 7 дней и `seasonal_week` — среднее отклонение `toy − trend7` по дням недели (Series с индексом 0–6, 1 знак). В `pattern_sum` — сумму `seasonal_week` (1 знак).

*Подсказки (2): `dsc.hint('toy-trend')`*

In [ ]:
%%task ch12/02-decompose/toy-trend
import pandas as pd

### Разности · разминка

Посчитайте для дневного ряда разность `d1 = daily.diff()` и недельную разность `d7 = daily.diff(7)`. Сохраните средние значения `mean_d1`, `mean_d7` (2 знака, без пропусков) и `std_ratio` — отношение стандартного отклонения `daily` к стандартному отклонению `d1` (2 знака).

*Подсказки (2): `dsc.hint('diff')`*

In [ ]:
%%task ch12/02-decompose/diff
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
daily = s.resample("D").sum()

### Тест на стационарность · база

Проверьте тестом Дики — Фуллера часовой ряд за август (`s["2026-08"]`), его суточную разность (`diff(24)`) и дневной ряд за всё время. Сохраните p-value `p_hour_aug`, `p_hour_diff24`, `p_daily` (4 знака) и список `stationary` — названия рядов (`"часы август"`, `"часы разность 24"`, `"дни"`), для которых p < 0.05.

*Подсказки (2): `dsc.hint('adf')`*

In [ ]:
%%task ch12/02-decompose/adf
import pandas as pd
from statsmodels.tsa.stattools import adfuller
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Сезонная поправка по часам · база

Разложите две недели (`s["2026-08-03":"2026-08-16"]`) с `period=24`. Сохраните `season24` — 24 значения сезонной части за первые сутки (Series с индексом-часом 0–23, 1 знак), `night_min` — час с наименьшей поправкой (если после округления до 1 знака минимумов несколько, подойдёт любой из них) и `amplitude` — разницу максимума и минимума (1 знак).

*Подсказки (2): `dsc.hint('hour-season')`*

In [ ]:
%%task ch12/02-decompose/hour-season
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Какая разность лучше · база

Для часового ряда посчитайте стандартные отклонения: `std_level` — самого ряда, `std_by_lag` — словарь `{1: …, 24: …, 168: …}` для разностей с этими лагами (все — 2 знака). Сохраните `best_lag` — лаг с наименьшим разбросом, и `gain` — во сколько раз разброс лучшей разности меньше разброса ряда (2 знака).

*Подсказки (2): `dsc.hint('seasonal-diff')`*

In [ ]:
%%task ch12/02-decompose/seasonal-diff
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Растёт ли размах · посложнее

Аддитивная модель предполагает, что суточные колебания одинаковы при любом уровне, мультипликативная — что они растут вместе с уровнем. Для марта и августа посчитайте средний суточный профиль (среднее по часу суток), его размах (`max − min`) и средний уровень (среднее заказов в час). Сохраните `amp` — словарь `{"03": размах, "08": размах}` (1 знак), `level` — то же для уровня (2 знака), `amp_growth` и `level_growth` — во сколько раз август больше марта (2 знака), и `model_type` — `"multiplicative"`, если отношения размахов и уровней отличаются меньше чем на 10 %, иначе `"additive"`.

*Подсказки (2): `dsc.hint('multiplicative')`*

In [ ]:
%%task ch12/02-decompose/multiplicative
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Автокорреляция по лагам · посложнее

Посчитайте автокорреляцию часового ряда для лагов 1…200 (`[s.autocorr(l) for l in range(1, 201)]`) и нарисуйте её столбцами (`ax.bar`), ось X — `"лаг, часов"`. Сохраните список `acf_vals`, `top3` — три лага с наибольшей автокорреляцией (по убыванию значения) и `near_day` — лаг с максимумом среди лагов 20–28.

*Подсказки (2): `dsc.hint('acf-plot')`*

In [ ]:
%%task ch12/02-decompose/acf-plot
import pandas as pd
import matplotlib.pyplot as plt
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Остаток показывает аномалии · со звёздочкой

По разложению дневного ряда (`period=7`) найдите дни, где остаток по модулю больше трёх стандартных отклонений остатка. Сохраните список дат `outliers` (как `datetime.date`, по возрастанию) и `kinds` — для каждой даты `"всплеск"`, если остаток положительный, иначе `"провал"`.

*Подсказки (2): `dsc.hint('residual-days')` · задача по желанию*

In [ ]:
%%task ch12/02-decompose/residual-days
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
daily = s.resample("D").sum()

## Урок 3. Признаки для прогноза ряда

Чтобы предсказать заказы в следующий час обычной моделью — линейной регрессией или бустингом, — ряд превращают в таблицу признаков: календарь, значения в прошлом (лаги), скользящие средние. Главное правило — в признаках только то, что известно на момент прогноза. Нарушить его здесь проще, чем кажется, поэтому в конце урока пишем автоматический тест на утечку. Как честно делить ряд на обучение и тест — в следующем уроке.

Теория — на сайте курса: `ch12/03-features.html`. Задачи «собери код из строк» (1) остались на сайте.

### Календарные признаки · разминка

Соберите DataFrame `df` со столбцами `y` (заказы), `hour`, `dow`, `weekend` (1 для субботы и воскресенья). Сохраните `weekend_share` — долю выходных часов (3 знака).

*Подсказки (2): `dsc.hint('calendar')`*

In [ ]:
%%task ch12/03-features/calendar
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Найти утечку · разминка

Ниже — четыре признака для строки «час t». Сохраните список `leaky` — названия тех, в которые попадают данные часа t или позже.

*Подсказки (2): `dsc.hint('leak-check')`*

In [ ]:
%%task ch12/03-features/leak-check
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
features = {
    "a": s.shift(1),
    "b": s.rolling(3).mean(),
    "c": s.shift(1).rolling(3).mean(),
    "d": s.shift(-1),
}

### Функция признаков · база

Напишите функцию `make_features(s)`, которая возвращает DataFrame без пропусков со столбцами: `y`, `hour`, `dow`, `weekend`, `lag_1`, `lag_24`, `lag_168`, `roll_24` (среднее за предыдущие 24 часа). Сохраните `df = make_features(s)` и `first_ts` — время первой строки.

*Подсказки (2): `dsc.hint('make-features')`*

In [ ]:
%%task ch12/03-features/make-features
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Какие признаки помогают · база

С помощью `make_features` из заготовки сравните `HistGradientBoostingRegressor(random_state=0)` на трёх наборах признаков: только календарь (`hour`, `dow`, `weekend`), календарь + лаги (`lag_1`, `lag_24`, `lag_168`), всё. Обучение — до 2026-08-01, тест — август. Сохраните словарь `rmse` (`{"календарь": ..., "лаги": ..., "всё": ...}`, 3 знака).

*Подсказки (2): `dsc.hint('feature-sets')`*

In [ ]:
%%task ch12/03-features/feature-sets
# уже есть: make_features(s) из прошлой задачи
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
df = make_features(s)
train = df.index < "2026-08-01"

### Автоматический тест на утечку · база

Функция `features(s)` из заготовки строит шесть признаков, и часть из них подглядывает в ответ. Три приёма в ней вам пока не встречались:

- `s.expanding().max()` — окно, которое начинается в начале ряда и растёт до текущей строки включительно: «максимум до сих пор»;
- `s.ewm(span=12).mean()` — экспоненциальное среднее: среднее по всему прошлому, где у свежих значений вес больше, а старые затухают; `span=12` задаёт скорость затухания: чем больше `span`, тем дольше помнится прошлое;
- `s.groupby(s.index.hour).transform(f)` — применяет `f` отдельно к ряду каждого часа суток (все 9:00 подряд, все 10:00 подряд…) и возвращает результат на места исходных строк.

Честны ли признаки по смыслу, определит ваш тест. Напишите `leak_test(features, s, t)`: испортите ряд в момент `t` (прибавьте 1000), постройте признаки по исходному и испорченному ряду и верните отсортированный список названий признаков, у которых значение **в строке `t`** изменилось. Примените к `t = pd.Timestamp("2026-07-15 12:00")` — результат в `leaking`.

*Подсказки (3): `dsc.hint('leak-test')`*

In [ ]:
%%task ch12/03-features/leak-test
import pandas as pd

def leak_test(features, s, t):
    ...

### Час как число и как категория · посложнее

Линейная регрессия на признаке «час» как числе не может описать суточный профиль с несколькими пиками. Сравните три линейные регрессии (обучение — до 1 августа, тест — август): `num` — признаки `hour` и `dow` числами; `onehot` — те же признаки, закодированные `pd.get_dummies(df[["hour", "dow"]].astype(str))`; `onehot_trend` — `onehot` плюс номер часа от начала ряда `t = np.arange(len(df))`. Сохраните `rmse` — словарь `{"num": …, "onehot": …, "onehot_trend": …}` (3 знака) и `n_onehot_cols` — число столбцов после кодирования.

*Подсказки (3): `dsc.hint('onehot-hours')`*

In [ ]:
%%task ch12/03-features/onehot-hours
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

def rmse(a, b):
    return round(float(np.sqrt(((a - b) ** 2).mean())), 3)

### Признак «день акции» · со звёздочкой

В уроке «Временной ряд: индекс-дата, resample и скользящие окна» (задача «Почему пятницы») выяснилось, что всплески — пятничные акции, о которых известно заранее: 17 апреля, 22 мая, 19 июня, 24 июля, 21 августа, с 10 до 20 часов. Добавьте к `make_features(s)` признак `promo` (1 в эти часы) и сравните RMSE бустинга на августе с ним и без него. Отдельно посчитайте RMSE только на часах акции 21 августа. Сохраните `rmse_without`, `rmse_with`, `promo_without`, `promo_with` (3 знака).

*Подсказки (2): `dsc.hint('promo-feature')` · задача по желанию*

In [ ]:
%%task ch12/03-features/promo-feature
# уже есть: make_features(s)
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
promo_days = pd.to_datetime(["2026-04-17", "2026-05-22", "2026-06-19", "2026-07-24", "2026-08-21"])

## Урок 4. Проверка по времени: хвост ряда и TimeSeriesSplit

Признаки без утечек — половина дела. Вторая половина — проверка: ряд нельзя делить случайно, иначе модель учится на часах после теста и оценка выходит оптимистичной. Разбираем честные способы — маску по дате, `train_test_split(shuffle=False)` и кросс-валидацию `TimeSeriesSplit` — и измеряем, насколько обманывает перемешивание.

Теория — на сайте курса: `ch12/03b-time-split.html`.

### Хвост ряда в тест · база

Таблица признаков `df = make_features(s)` готова. Отложите последние 20 % строк функцией `train_test_split(X, y, test_size=0.2, shuffle=False)`, обучите `HistGradientBoostingRegressor(random_state=0)` и сохраните `test_start` — время первой тестовой строки (`Timestamp`) и `rmse_time` — RMSE на тесте (3 знака). Затем повторите с перемешиванием (`random_state=0` вместо `shuffle=False`) — `rmse_shuffled` (3 знака). В `gap` сохраните, на сколько перемешанная оценка оптимистичнее: `rmse_time - rmse_shuffled` (3 знака).

*Подсказки (3): `dsc.hint('noshuffle')`*

In [ ]:
%%task ch12/03b-time-split/noshuffle
# уже есть: make_features(s)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
df = make_features(s)
X, y = df.drop(columns="y"), df["y"]

### Границы фолдов · разминка

Разбейте часовой ряд `s` с помощью `TimeSeriesSplit(n_splits=4)`. Сохраните `train_sizes` — список длин обучающей части в каждом фолде и `test_starts` — список первых тестовых часов (значения индекса `s`). Проверьте себя: каждый тест должен начинаться ровно через час после конца своего обучения.

*Подсказки (2): `dsc.hint('ch12/03b-time-split/folds')`*

In [ ]:
%%task ch12/03b-time-split/folds
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Кросс-валидация по времени · посложнее

Оцените бустинг на всех признаках `make_features(s)` кросс-валидацией `TimeSeriesSplit(n_splits=5)` по RMSE (`scoring="neg_root_mean_squared_error"`). Сохраните `scores` — RMSE на фолдах (положительные числа, 2 знака) и `mean_rmse` (3 знака). Затем посчитайте то же с обычным `KFold(5, shuffle=True, random_state=0)` — `mean_rmse_kfold` (3 знака).

*Подсказки (2): `dsc.hint('tscv')`*

In [ ]:
%%task ch12/03b-time-split/tscv
# уже есть: make_features(s)
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold, cross_val_score
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
df = make_features(s)
X, y = df.drop(columns="y"), df["y"]

## Урок 5. Прогноз: ориентиры, горизонт и метрики

Прежде чем хвалить модель, сравните её с простыми прогнозами вроде «как неделю назад» — у рядов они бывают пугающе сильными. Разбираемся с метриками прогноза, с тем, почему MAPE ломается на нулях, и с горизонтом: прогноз на час вперёд и на завтра — разные задачи с разными признаками.

Теория — на сайте курса: `ch12/04-forecast.html`. Задачи «собери код из строк» (1) остались на сайте.

### Наивные прогнозы · разминка

Посчитайте на тесте (август) MAE трёх наивных прогнозов: `s.shift(1)`, `s.shift(24)`, `s.shift(168)`. Сохраните словарь `mae` (`{1: ..., 24: ..., 168: ...}`, 3 знака) и `best_lag`.

*Подсказки (2): `dsc.hint('naive')`*

In [ ]:
%%task ch12/04-forecast/naive
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
test = s["2026-08-01":]

### Метрики своими руками · разминка

Напишите функции `mae(fact, pred)`, `rmse(fact, pred)` и `wape(fact, pred)` для массивов numpy. Посчитайте их для `fact` и `pred` из заготовки: `m` — словарь `{"mae": …, "rmse": …, "wape": …}` (3 знака).

*Подсказки (2): `dsc.hint('metrics')`*

In [ ]:
%%task ch12/04-forecast/metrics
import numpy as np

def mae(fact, pred):
    ...

def rmse(fact, pred):
    ...

def wape(fact, pred):
    ...

### WAPE против MAPE · разминка

Напишите функцию `wape(fact, pred)`. Для прогноза «неделю назад» на августе сохраните `w` (3 знака) и `zeros` — число часов теста с нулевым фактом (на них MAPE не определена).

*Подсказки (2): `dsc.hint('wape')`*

In [ ]:
%%task ch12/04-forecast/wape
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
test = s["2026-08-01":]
pred = s.shift(168)[test.index]

### Систематический недопрогноз · база

У прогноза бывает не только разброс, но и **смещение** — средняя ошибка со знаком. Для прогноза «среднее за 4 недели» (`m4` в заготовке) на августе посчитайте `bias` — среднюю ошибку `факт − прогноз` (3 знака), `share_under` — долю часов, где прогноз меньше факта (3 знака), и `ratio` — сумма факта / сумма прогноза (3 знака). В заготовке есть и поправленный прогноз `adj`: `m4`, умноженный на коэффициент `_k` — во сколько раз факт за последнюю неделю выше прогноза `m4` за ту же неделю. Если прогноз отстаёт от роста, `_k > 1` и подтягивает его вверх. Для `adj` — посчитайте для него `bias_adj` и сравните RMSE: `rmse_m4`, `rmse_adj` (3 знака).

*Подсказки (3): `dsc.hint('bias')`*

In [ ]:
%%task ch12/04-forecast/bias
import numpy as np

### Сильный наивный прогноз · база

Постройте прогноз «среднее за этот час в k прошлых недель» для k = 1, 2, 4, 8 и посчитайте RMSE на августе. Сохраните словарь `rmse_k` (3 знака) и `best_k`. Нужная история для k = 8 есть: тест начинается после 22 недель данных.

*Подсказки (2): `dsc.hint('mean4w')`*

In [ ]:
%%task ch12/04-forecast/mean4w
import numpy as np
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]
test = s["2026-08-01":]

### Цена горизонта · база

Сравните бустинг `HistGradientBoostingRegressor(random_state=0)` для двух горизонтов на признаках из теории (функция `make` в заготовке): на час вперёд и на завтра. Здесь, как в теории, отсечка — полночь, поэтому лаг 24 для завтрашних часов законен; с решением в 20:00 разберёмся в проекте главы. Сохраните `rmse_hour`, `rmse_day` (3 знака) и `gap_pct` — на сколько процентов RMSE на завтра больше (1 знак).

*Подсказки (2): `dsc.hint('horizon-gap')`*

In [ ]:
%%task ch12/04-forecast/horizon-gap
# уже есть: make(s, lags, roll_shift) из теории
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Утечка из-за горизонта · посложнее

Проверьте, что будет, если для прогноза «на завтра» (отсечка — полночь, как в теории) по ошибке оставить лаг 1. Обучите `LinearRegression` на признаках `make(s, (1, 24, 48, 168), 1)` и на честных `make(s, (24, 48, 168), 24)`. Сохраните `rmse_leak`, `rmse_fair` (3 знака) и `fooled` — `True`, если «прогноз с утечкой» выглядит лучше.

*Подсказки (2): `dsc.hint('leak-horizon')`*

In [ ]:
%%task ch12/04-forecast/leak-horizon
# уже есть: make(s, lags, roll_shift)
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Где модель ошибается · со звёздочкой

Для честного прогноза «на завтра» с отсечкой в полночь (`LinearRegression`, признаки `make(s, (24, 48, 168), 24)`) посчитайте MAE по часам суток на тесте. Сохраните Series `mae_by_hour` (индекс 0–23, 2 знака), `worst_hour` и `night_share` — долю суммарной абсолютной ошибки, приходящуюся на часы 0–5 (3 знака).

*Подсказки (2): `dsc.hint('by-hour-error')` · задача по желанию*

In [ ]:
%%task ch12/04-forecast/by-hour-error
# уже есть: make(s, lags, roll_shift)
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

## Урок 6. Проект: сколько бариста поставить на смену

Финал главы. Заказы из приложения собирает отдельный бариста на выдаче, и график смен составляется накануне. Строим прогноз заказов на завтра по часам с требованием к точности, честно выбираем модель на валидации — и переводим прогноз в людей, где выясняется, что «точный в среднем» прогноз систематически оставляет смены без рук.

Теория — на сайте курса: `ch12/05-project.html`.

### Шаг 1. Разбиение по времени · база

Постройте `df = make(s)`, `X`, `y` и три маски: `train` (до 2026-07-01), `valid` (июль), `test` (август). Сохраните `sizes` — словарь `{"train": ..., "valid": ..., "test": ...}` с числом строк.

*Подсказки (3): `dsc.hint('ch12/05-project/split')`*

In [ ]:
%%task ch12/05-project/split
# уже есть: make(s) — таблица признаков для прогноза на завтра
import pandas as pd
s = pd.read_csv("app_orders.csv", parse_dates=["datetime"], index_col="datetime")["orders"]

### Шаг 2. Ориентир и требование · база

Посчитайте RMSE прогноза «среднее за 4 недели» (столбец `mean4w`) на тесте — `rmse_base` (3 знака). В `base_meets` сохраните, проходит ли ориентир требование «RMSE ≤ 4.3».

*Подсказки (2): `dsc.hint('ch12/05-project/baseline')`*

In [ ]:
%%task ch12/05-project/baseline
# уже есть: df, X, y и маски train, valid, test
import numpy as np

### Шаг 3. Выбор на валидации · посложнее

Обучите на `train` и проверьте на `valid` четыре модели: `LinearRegression()`, `HistGradientBoostingRegressor(random_state=0)`, `HistGradientBoostingRegressor(max_depth=3, random_state=0)`, `HistGradientBoostingRegressor(l2_regularization=1.0, random_state=0)`. Сохраните словарь `valid_rmse` с ключами `"linear"`, `"hgb"`, `"hgb_depth3"`, `"hgb_l2"` (3 знака) и `best` — ключ лучшей.

*Подсказки (2): `dsc.hint('select')`*

In [ ]:
%%task ch12/05-project/select
# уже есть: df, X, y и маски train, valid, test
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor

### Шаг 3б. Подбор параметров по времени · посложнее

В шаге 3 модель выбрана на одном месяце (июль): впереди неглубокий бустинг, но его параметры взяты наугад. Надёжнее проверить параметры на нескольких кусках прошлого — кросс-валидацией **по времени**. Подберите параметры бустинга на `train | valid` (март–июль) — найденные `best_params` станут параметрами финальной модели в шаге 4:

```python static
GridSearchCV(HistGradientBoostingRegressor(random_state=0),
             {"max_depth": [2, 3, None], "learning_rate": [0.05, 0.1]},
             cv=TimeSeriesSplit(n_splits=3), scoring="neg_root_mean_squared_error")
```

Сохраните обученный объект в `gs`, лучшие параметры — в `best_params` (словарь), их RMSE на кросс-валидации — `cv_rmse` (положительное число, 3 знака). Затем посчитайте RMSE лучшей модели на августе с обрезкой нулём — `rmse_test_grid` (3 знака) и сравните с требованием 4.3.

*Подсказки (3): `dsc.hint('ch12/05-project/grid')`*

In [ ]:
%%task ch12/05-project/grid
# уже есть: df, X, y и маски train, valid, test
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

fit_mask = train | valid

### Шаг 4. Финальная проверка · база

В заготовке `best_params` — параметры бустинга, найденные по времени в шаге 3б. Обучите `HistGradientBoostingRegressor(random_state=0, **best_params)` на `train | valid`, обрежьте прогноз снизу нулём (`np.clip(p, 0, None)`) и посчитайте RMSE на тесте — `rmse_final` (3 знака). Сохраните `pred_test` (массив прогнозов) и `meets` — выполнено ли требование целиком: RMSE ≤ 4.3 **и** лучше ориентира `rmse_base`.

*Подсказки (2): `dsc.hint('ch12/05-project/final')`*

In [ ]:
%%task ch12/05-project/final
# уже есть: X, y, маски train/valid/test, rmse_base и best_params из шага 3б
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

### Шаг 5. Прогноз в людях · посложнее

Переведите прогноз в бариста: нужно `max(1, ceil(заказы / 6))` человек в час (с 7 до 22 включительно; ночью выдачу не планируем). Посчитайте для теста:

- `needed` — сколько бариста нужно было по факту;
- план по модели (`pred_test` в заготовке) и план по ориентиру (`mean4w`).

Для каждого плана сохраните в словарь `plans` (`{"модель": {...}, "ориентир": {...}}`) долю часов с нехваткой людей `short` (план < нужного, 3 знака) и число лишних бариста-часов `extra` (сумма положительных `план − нужно`).

*Подсказки (2): `dsc.hint('staffing')`*

In [ ]:
%%task ch12/05-project/staffing
# уже есть: X, y, маски и pred_test — прогноз модели на август
import numpy as np

y_test = y[test]
open_hours = (y_test.index.hour >= 7) & (y_test.index.hour <= 22)

### Шаг 6. План по верхнему квантилю · со звёздочкой

Если нехватка рук дороже лишней смены, планируют не по середине, а по **верхнему квантилю**: значение, которое факт превышает лишь в ~20 % случаев. Обучите `HistGradientBoostingRegressor(loss="quantile", quantile=0.8, random_state=0, **best_params)` (параметры из шага 3б есть в заготовке) на `train | valid`, обрежьте нулём и постройте план как в шаге 5. Сохраните `coverage` — долю часов теста, где факт не выше прогноза (3 знака), `short_q` (доля часов нехватки, 3 знака) и `extra_q` (лишние бариста-часы) — по рабочим часам.

*Подсказки (2): `dsc.hint('ch12/05-project/quantile')` · задача по желанию*

In [ ]:
%%task ch12/05-project/quantile
# уже есть: X, y, маски train, valid, test и best_params
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()